### Main notebook to fetch everything

In [1]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import wandb

In [2]:
# 1. Setup output paths
REPORTS_DIR = Path("reports")
FIG_DIR = REPORTS_DIR / "figures"
TABLE_DIR = REPORTS_DIR / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

In [20]:
# 2. Connect to WandB API
api = wandb.Api()

In [21]:
# Auto-detect your default WandB username/entity
PROJECT_NAME = "multimodal-brain-tumor-diagnostics"
try:
    ENTITY = api.default_entity
except Exception:
    ENTITY = api.viewer["entity"]


# Safely fetch runs
runs = api.runs(f"{ENTITY}/{PROJECT_NAME}")



In [22]:
# 3. Pull summary metrics across all runs
run_data = []
for run in runs:
    # Filter or identify model type based on run tags/name
    run_info = {
        "Run ID": run.id,
        "Model Name": run.name,
        "Architecture": run.config.get("architecture", "U-Net"),
        "Task Type": run.config.get("task_type", "MTL"),
        "Best Seg Dice (A)": run.summary.get("val_dice", 0.0),
        "Best Cls Acc (B)": run.summary.get("val_accuracy", 0.0),
        "Best Cls AUC": run.summary.get("val_auc", 0.0),
        "Final Loss": run.summary.get("val_loss", 0.0),
    }
    run_data.append(run_info)
df_summary = pd.DataFrame(run_data)

In [23]:
# Save to reports/tables/
csv_path = TABLE_DIR / "experiment_summary.csv"
tex_path = TABLE_DIR / "experiment_summary.tex"

df_summary.to_csv(csv_path, index=False)
df_summary.to_latex(tex_path, index=False, float_format="%.4f")

print(
    f"✅ Saved experiment table with {len(df_summary)} runs to:\n   - {csv_path}\n   - {tex_path}"
)
df_summary

✅ Saved experiment table with 1 runs to:
   - reports\tables\experiment_summary.csv
   - reports\tables\experiment_summary.tex


,Run ID,Model Name,Architecture,Task Type,Best Seg Dice (A),Best Cls Acc (B),Best Cls AUC,Final Loss
0,8n1prhi7,solar-dragon-34,U-Net,MTL,0.726291,0.0,0.0,0.461351
